In [0]:
# Databricks notebook source
import os
from databricks.sdk import WorkspaceClient

# Obtener parámetros de widgets
dbutils.widgets.text("catalog", "datathon")
dbutils.widgets.text("schema", "gcc")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# 1. Asegurar catálogo, esquema y volumen en Unity Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.raw")

target_volume_dir = f"/Volumes/{catalog}/{schema}/raw"
target_path = os.path.join(target_volume_dir, "datathon_data_gcc.csv")

# 2. Si el archivo ya existe y tiene datos en el volumen, omitir copia
if os.path.exists(target_path) and os.path.getsize(target_path) > 0:
    print(f"El dataset ya esta disponible en el volumen: {target_path}")
    print("Omitiendo copia para evitar sobreescritura.")
else:
    print(f"Preparando archivo en Unity Catalog Volume: {target_path}")
    
    # Resolver ruta del archivo en el workspace
    current_dir = os.getcwd()
    source_path = os.path.normpath(os.path.join(current_dir, "..", "data", "datathon_data_gcc.csv"))
    
    if not os.path.exists(source_path):
        alt_path = os.path.join(current_dir, "data", "datathon_data_gcc.csv")
        if os.path.exists(alt_path):
            source_path = alt_path

    # Formatear ruta para la API del Workspace (remueve el prefijo /Workspace si existe)
    api_workspace_path = source_path
    if api_workspace_path.startswith("/Workspace"):
        api_workspace_path = api_workspace_path[len("/Workspace"):]

    print(f"Transfiriendo archivo via Databricks WorkspaceClient: {api_workspace_path}")
    
    try:
        w = WorkspaceClient()
        downloaded = w.workspace.download(api_workspace_path)
        with open(target_path, "wb") as f_out:
            f_out.write(downloaded.read())
        print(f"Archivo copiado exitosamente al volumen: {target_path}")
    except Exception as e:
        print(f"Fallo en descarga por WorkspaceClient: {e}")
        # Si por alguna razon falla la API pero el archivo ya existia en el destino
        if not (os.path.exists(target_path) and os.path.getsize(target_path) > 0):
            raise e

print("Setup de infraestructura y datos finalizado con exito.")